# Model 2 — Binary Logistic Regression dengan Efek Interaksi
## Pemilihan Hotel: Novotel (pasca-merger) vs Pullman

Model 1 (sederhana, tanpa interaksi) sudah signifikan secara keseluruhan, tapi
**Hosmer-Lemeshow test menunjukkan bentuk model belum fit** dengan baik. Notebook ini
membangun **Model 2** dengan menambahkan efek interaksi **Income_JutaRp x TujuanMenginap_Bisnis**
— sesuai hint di soal: sensitivitas income terhadap pilihan brand kemungkinan berbeda
antara tamu bisnis (biaya di-reimburse kantor) dan leisure/pribadi (biaya ditanggung sendiri).

Model 1 dan Model 2 lalu dibandingkan secara statistik (Likelihood Ratio test antar-model,
AIC/BIC) maupun bisnis, untuk memilih spesifikasi final (poin B.7).


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from logit_diagnostics import full_diagnostic_report, validation_report, train_fit_report, train_test_comparison


## 1. Load Data & Operasionalisasi Variabel (sama seperti Model 1, + 1 variabel interaksi)

In [2]:
df = pd.read_csv('Data_Binary_Logit_Pullman-vs-Novotel.csv')

df['BrandPilihan_Pullman'] = (df['BrandPilihan'] == 'Pullman').astype(int)
df['TujuanMenginap_Bisnis'] = (df['TujuanMenginap'] == 'Bisnis').astype(int)

# Variabel interaksi: efek Income diperbolehkan berbeda antara tamu Bisnis vs Leisure
df['Income_x_Bisnis'] = df['Income_JutaRp'] * df['TujuanMenginap_Bisnis']

feature_cols_m1 = [
    'TujuanMenginap_Bisnis', 'Income_JutaRp', 'Harga_Novotel',
    'Harga_Pullman', 'Iklan_Novotel', 'Iklan_Pullman',
]
feature_cols_m2 = feature_cols_m1 + ['Income_x_Bisnis']

y = df['BrandPilihan_Pullman']
X1 = df[feature_cols_m1]
X2 = df[feature_cols_m2]

df[['BrandPilihan_Pullman', 'TujuanMenginap_Bisnis', 'Income_JutaRp', 'Income_x_Bisnis']].head()


,BrandPilihan_Pullman,TujuanMenginap_Bisnis,Income_JutaRp,Income_x_Bisnis
0,0,1,16.7,16.7
1,1,1,11.2,11.2
2,0,0,20.6,0.0
3,1,1,3.0,3.0
4,0,0,17.3,0.0


## 2. Split Data (Sama Persis dengan Model 1)

Menggunakan `random_state=42` yang sama seperti Model 1, supaya baris data training/testing
identik dan perbandingan kedua model benar-benar apple-to-apple.


In [3]:
X1_train, X1_test, y_train, y_test = train_test_split(
    X1, y, test_size=0.2, random_state=42, stratify=y
)
X2_train, X2_test, _, _ = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X2_train.shape, ' | Test:', X2_test.shape)


Train: (400, 7)  | Test: (100, 7)


## 3. Estimasi Model 1 (Ulang, sbg Pembanding) dan Model 2 (dengan Interaksi)

In [4]:
model1 = sm.Logit(y_train, sm.add_constant(X1_train)).fit(disp=0)
model2 = sm.Logit(y_train, sm.add_constant(X2_train)).fit(disp=0)

print('Model 1 (tanpa interaksi):')
print(f'  Log-Likelihood = {model1.llf:.4f} | AIC = {model1.aic:.2f} | BIC = {model1.bic:.2f} | McFadden R2 = {model1.prsquared:.4f}')
print('\nModel 2 (dengan interaksi Income x Bisnis):')
print(f'  Log-Likelihood = {model2.llf:.4f} | AIC = {model2.aic:.2f} | BIC = {model2.bic:.2f} | McFadden R2 = {model2.prsquared:.4f}')


Model 1 (tanpa interaksi):
  Log-Likelihood = -203.7967 | AIC = 421.59 | BIC = 449.53 | McFadden R2 = 0.2648

Model 2 (dengan interaksi Income x Bisnis):
  Log-Likelihood = -166.1577 | AIC = 348.32 | BIC = 380.25 | McFadden R2 = 0.4006


## 4. Membandingkan Model 1 vs Model 2 — Likelihood Ratio Test Antar-Model

Karena Model 1 adalah **nested model** dari Model 2 (Model 1 = Model 2 dengan koefisien
interaksi dipaksa nol), kita bisa uji formal apakah menambah variabel interaksi
memperbaiki model secara signifikan, menggunakan LR test antar dua model (bukan LR test vs
null seperti pada bagian overall significance masing-masing model).

H0: koefisien interaksi = 0 (Model 1 sudah cukup)
H1: koefisien interaksi != 0 (Model 2 signifikan lebih baik)


In [5]:
lr_stat = 2 * (model2.llf - model1.llf)
df_diff = model2.df_model - model1.df_model
lr_pvalue = 1 - stats.chi2.cdf(lr_stat, df_diff)

print(f'LR statistic       : {lr_stat:.4f}')
print(f'df (selisih param) : {int(df_diff)}')
print(f'p-value            : {lr_pvalue:.10f}')

if lr_pvalue < 0.05:
    print('\n-> Signifikan: penambahan efek interaksi Income x Bisnis MEMPERBAIKI model secara nyata.')
    print('   Model 2 lebih disukai dibanding Model 1 (tolak H0).')
else:
    print('\n-> Tidak signifikan: efek interaksi tidak menambah nilai statistik yang berarti.')


LR statistic       : 75.2779
df (selisih param) : 1
p-value            : 0.0000000000

-> Signifikan: penambahan efek interaksi Income x Bisnis MEMPERBAIKI model secara nyata.
   Model 2 lebih disukai dibanding Model 1 (tolak H0).


In [6]:
comparison = pd.DataFrame({
    'Model 1 (tanpa interaksi)': [model1.llf, model1.aic, model1.bic, model1.prsquared, model1.df_model],
    'Model 2 (dengan interaksi)': [model2.llf, model2.aic, model2.bic, model2.prsquared, model2.df_model],
}, index=['Log-Likelihood', 'AIC', 'BIC', 'McFadden R2', 'Jumlah Prediktor'])
comparison.round(4)


,Model 1 (tanpa interaksi),Model 2 (dengan interaksi)
Log-Likelihood,-203.7967,-166.1577
AIC,421.5933,348.3154
BIC,449.5336,380.2471
McFadden R2,0.2648,0.4006
Jumlah Prediktor,6.0000,7.0000


**Catatan AIC/BIC:** semakin kecil nilainya, semakin baik model (mempertimbangkan trade-off
antara kecocokan model dan kompleksitas/jumlah parameter). Model dengan AIC/BIC lebih rendah
lebih disukai meski jumlah variabelnya lebih banyak, karena penambahan itu sepadan dengan
perbaikan log-likelihood-nya.

## 5. Full Diagnostic Report — Model 2 (Training)

In [7]:
model2, results2 = full_diagnostic_report(
    X2_train, y_train,
    feature_names=feature_cols_m2,
    y_name='BrandPilihan_Pullman',
    model_label='Model 2 (With Interaction: Income x Bisnis)',
    labels=('Novotel', 'Pullman'),
)



######################################################################
#  FULL DIAGNOSTIC REPORT (Binary Logit) - Model 2 (With Interaction: Income x Bisnis)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                       TujuanMenginap_Bisnis  Income_JutaRp  Harga_Novotel  Harga_Pullman  Iklan_Novotel  Iklan_Pullman  Income_x_Bisnis
TujuanMenginap_Bisnis                  1.000          0.050         -0.037          0.000          0.088          0.037            0.772
Income_JutaRp                          0.050          1.000         -0.001          0.017          0.043         -0.001            0.470
Harga_Novotel                         -0.037         -0.001          1.000          0.023         -0.023          0.072            0.016
Harga_Pullman                          0.000          0.017          0.023          1.000          0.115         -0.065            0.017
Iklan_Novotel             

## 6. Ringkasan statsmodels (opsional, untuk cross-check)

In [8]:
print(model2.summary())


                            Logit Regression Results                            
Dep. Variable:     BrandPilihan_Pullman   No. Observations:                  400
Model:                            Logit   Df Residuals:                      392
Method:                             MLE   Df Model:                            7
Date:                  Sun, 13 Sep 2026   Pseudo R-squ.:                  0.4006
Time:                          08:06:42   Log-Likelihood:                -166.16
converged:                         True   LL-Null:                       -277.21
Covariance Type:              nonrobust   LLR p-value:                 2.350e-44
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -7.0093      1.607     -4.361      0.000     -10.160      -3.859
TujuanMenginap_Bisnis     7.5866      1.008      7.530      0.000       5.612     

## 7. Fit di Data Training vs Validasi Hold-Out (20%) — Model 2

In [9]:
train_result2 = train_fit_report(model2, X2_train, y_train, label='Training Set (80%) - Model 2',
                                  labels=('Novotel', 'Pullman'))


FIT MODEL DI DATA TRAINING - Training Set (80%) - Model 2
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel           140            57
Actual Pullman            21           182

N                  : 400
Hit Rate (Akurasi) : 0.8050  (80.50%)
Sensitivity (Recall Pullman) : 0.8966
Specificity (Recall Novotel) : 0.7107
Precision (Pullman)          : 0.7615

Proportional Chance Criterion : 0.5001 (hit rate model sebaiknya > 1.25x ini = 0.6251, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 148.8400  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [10]:
test_result2 = validation_report(model2, X2_test, y_test, label='Hold-Out / Test Set (20%) - Model 2',
                                  labels=('Novotel', 'Pullman'))


VALIDASI MODEL - Hold-Out / Test Set (20%) - Model 2
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel            31            18
Actual Pullman             9            42

N                  : 100
Hit Rate (Akurasi) : 0.7300  (73.00%)
Sensitivity (Recall Pullman) : 0.8235
Specificity (Recall Novotel) : 0.6327
Precision (Pullman)          : 0.7000

Proportional Chance Criterion : 0.5002 (hit rate model sebaiknya > 1.25x ini = 0.6252, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 21.1600  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [11]:
train_test_comparison(train_result2, test_result2)


RINGKASAN PERBANDINGAN TRAIN vs TEST
Hit Rate         Train      Test         Gap
                0.8050    0.7300     +0.0750  <- indikasi overfitting



## 8. Interpretasi Efek Interaksi — Kenapa Ini Penting?

Karena `Income_x_Bisnis` signifikan, koefisien `Income_JutaRp` **tidak boleh dibaca sendirian**.
Efek income terhadap log-odds memilih Pullman sekarang **berbeda tergantung TujuanMenginap**:

- Untuk tamu **Leisure/Pribadi** (Bisnis=0): efek income = koefisien `Income_JutaRp` saja
- Untuk tamu **Bisnis** (Bisnis=1): efek income = koefisien `Income_JutaRp` + koefisien `Income_x_Bisnis`


In [12]:
b_income = model2.params['Income_JutaRp']
b_interaction = model2.params['Income_x_Bisnis']

slope_leisure = b_income
slope_bisnis = b_income + b_interaction

effect_table = pd.DataFrame({
    'Segmen': ['Leisure/Pribadi', 'Bisnis'],
    'Slope Income (log-odds per Rp1jt)': [slope_leisure, slope_bisnis],
    'Odds Ratio per Rp1jt Income': [np.exp(slope_leisure), np.exp(slope_bisnis)],
    'Kenaikan Odds (%)': [(np.exp(slope_leisure)-1)*100, (np.exp(slope_bisnis)-1)*100],
})
effect_table.round(4)


,Segmen,Slope Income (log-odds per Rp1jt),Odds Ratio per Rp1jt Income,Kenaikan Odds (%)
0,Leisure/Pribadi,0.3395,1.4042,40.4184
1,Bisnis,0.0058,1.0058,0.5779


**Insight bisnis:** untuk tamu **leisure/pribadi**, setiap kenaikan Rp1 juta income
menaikkan odds memilih Pullman cukup besar (income betul-betul mendorong ke brand premium
karena mereka membayar sendiri). Sebaliknya, untuk tamu **bisnis**, efek income terhadap
pilihan brand nyaris hilang (slope mendekati 0) — masuk akal karena biaya perjalanan bisnis
di-reimburse kantor, sehingga income pribadi tamu tidak banyak berpengaruh pada pilihan
brand. **Inilah pola yang tidak bisa ditangkap Model 1** (yang memaksa satu slope income
yang sama untuk semua tamu), dan penjelasannya persis sesuai hint di soal kasus.


## 9. Kesimpulan Pemilihan Model

Ringkasan cepat untuk menjawab poin B.7 (perbandingan minimal dua spesifikasi model).


In [13]:
summary_rows = {
    'LR test antar-model (Model1 vs Model2)': f'chi2={lr_stat:.2f}, p={lr_pvalue:.2e} -> Model 2 signifikan lebih baik' if lr_pvalue < 0.05 else 'Tidak signifikan',
    'AIC': f'Model1={model1.aic:.2f} vs Model2={model2.aic:.2f} -> {"Model 2 lebih baik" if model2.aic < model1.aic else "Model 1 lebih baik"}',
    'BIC': f'Model1={model1.bic:.2f} vs Model2={model2.bic:.2f} -> {"Model 2 lebih baik" if model2.bic < model1.bic else "Model 1 lebih baik"}',
    'McFadden R2': f'Model1={model1.prsquared:.4f} vs Model2={model2.prsquared:.4f}',
    'Hosmer-Lemeshow fit': f"Model1 p={results2['hosmer_lemeshow']['hl_pvalue']:.4f} (cek notebook Model 1 utk p Model1)",
    'Hit rate hold-out': f'Model2 = {test_result2["classification"]["hit_rate"]*100:.1f}%',
}
for k, v in summary_rows.items():
    print(f'- {k}: {v}')

print('\n=> KESIMPULAN: Model 2 (dengan interaksi Income x TujuanMenginap) dipilih sebagai')
print('   model final — signifikan lebih baik secara statistik (LR test, AIC, BIC lebih rendah,')
print('   McFadden R2 lebih tinggi, Hosmer-Lemeshow lebih fit) dan lebih masuk akal secara bisnis')
print('   karena menangkap perbedaan sensitivitas income antara tamu bisnis vs leisure.')


- LR test antar-model (Model1 vs Model2): chi2=75.28, p=0.00e+00 -> Model 2 signifikan lebih baik
- AIC: Model1=421.59 vs Model2=348.32 -> Model 2 lebih baik
- BIC: Model1=449.53 vs Model2=380.25 -> Model 2 lebih baik
- McFadden R2: Model1=0.2648 vs Model2=0.4006
- Hosmer-Lemeshow fit: Model1 p=0.2351 (cek notebook Model 1 utk p Model1)
- Hit rate hold-out: Model2 = 73.0%

=> KESIMPULAN: Model 2 (dengan interaksi Income x TujuanMenginap) dipilih sebagai
   model final — signifikan lebih baik secara statistik (LR test, AIC, BIC lebih rendah,
   McFadden R2 lebih tinggi, Hosmer-Lemeshow lebih fit) dan lebih masuk akal secara bisnis
   karena menangkap perbedaan sensitivitas income antara tamu bisnis vs leisure.
